# 8.3 Tokenization (BPE, WordPiece, SentencePiece)

Subword tokenization is the compromise that lets text stay open-vocabulary without making every character its own word.

BPE greedily merges frequent adjacent symbols, so common fragments become one token while rare forms can still be assembled. The vocabulary and tie-breaking policy become part of the model contract.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): BPE pair counts and merges

BPE chooses the next merge by $$(x^*,y^*)=\arg\max_{(x,y)}\sum_{w\in V}\operatorname{count}_w(x,y).$$ On the lesson words `low`, `lower`, `newest`, and `widest`, several pairs reach count 2; deterministic first-seen tie-breaking makes `(l,o)` the first merge.

In [ ]:

def word_to_symbols(word):
    return tuple(list(word) + ["_"])


def count_pairs(words):
    counts = Counter()
    first_seen = {}
    position = 0
    for symbols in words:
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            counts[pair] += 1
            if pair not in first_seen:
                first_seen[pair] = position
            position += 1
    return counts, first_seen


def merge_pair_in_word(symbols, pair):
    merged = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            merged.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return tuple(merged)

lesson_words = [word_to_symbols(word) for word in ["low", "lower", "newest", "widest"]]
pair_counts, first_seen = count_pairs(lesson_words)
first_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], -first_seen[pair]))

assert pair_counts[("l", "o")] == 2
assert pair_counts[("o", "w")] == 2
assert pair_counts[("e", "s")] == 2
assert first_pair == ("l", "o")


The reusable trainer records the merge list, applies each merge everywhere, and then tokenizes new words with the frozen training merges.

In [ ]:

def train_bpe(corpus, num_merges):
    words = [word_to_symbols(word) for word in corpus]
    merges = []
    snapshots = [words]
    for step in range(num_merges):
        counts, first_seen = count_pairs(words)
        if not counts:
            break
        best_pair = max(counts, key=lambda pair: (counts[pair], -first_seen[pair]))
        merges.append(best_pair)
        words = [merge_pair_in_word(symbols, best_pair) for symbols in words]
        snapshots.append(words)
    return merges, snapshots


def apply_bpe(word, merges, continuation=False):
    symbols = word_to_symbols(word)
    for pair in merges:
        symbols = merge_pair_in_word(symbols, pair)
    pieces = [piece for piece in symbols if piece != "_"]
    if continuation:
        marked = []
        for i, piece in enumerate(pieces):
            marker = "" if i == 0 else "##"
            marked.append(marker + piece.replace("_", ""))
        return marked
    return [piece.replace("_", "") for piece in pieces]

merges, snapshots = train_bpe(["low", "lower", "newest", "widest"], 1)
low_before = list(word_to_symbols("low"))
low_after = list(apply_bpe("low", merges))

assert merges[0] == ("l", "o")
assert len(low_before) == 4
assert len(low_after) == 3
assert 4 - len(low_after) == 1


## Dataset ladder: inline subword corpora from toy words to longer multilingual-looking strings

In [ ]:

bpe_ladder = [
    {
        "name": "D1 lesson BPE words",
        "corpus": ["low", "lower", "newest", "widest"],
        "eval": ["lowest", "lower"],
        "merges": 4,
    },
    {
        "name": "D2 five clean invented words",
        "corpus": ["adview", "adclick", "adplay", "bidview", "bidclick"],
        "eval": ["adclick", "bidplay", "adview"],
        "merges": 6,
    },
    {
        "name": "D3 tie-breaking and unknown morphology",
        "corpus": ["play", "played", "playing", "player", "replay"],
        "eval": ["playful", "replayed", "playing"],
        "merges": 8,
    },
    {
        "name": "D4 inline product search corpus",
        "corpus": ["campaign", "manager", "creative", "creator", "marketplace", "invoice", "preview"],
        "eval": ["campaigns", "creatives", "marketplaces"],
        "merges": 12,
    },
    {
        "name": "D5 longer multilingual-looking strings",
        "corpus": ["cafepromo", "saopaulo", "resumeupload", "marktanalyse", "creadorcampana", "facturacion"],
        "eval": ["cafepromos", "saopauloinvoice", "creadorcampanas", "resumeuploads"],
        "merges": 16,
    },
]

for rung in bpe_ladder:
    print(rung["name"], "train words", len(rung["corpus"]), "eval words", len(rung["eval"]))
    print("sample", rung["corpus"][0])


In [ ]:

bpe_results = []
for index, rung in enumerate(bpe_ladder, start=1):
    merges, snapshots = train_bpe(rung["corpus"], rung["merges"])
    lengths = [len(apply_bpe(word, merges, continuation=True)) for word in rung["eval"]]
    avg_len = float(np.mean(lengths))
    bpe_results.append({"rung": index, "name": rung["name"], "avg_tokens": avg_len, "merges": merges})

for row in bpe_results:
    print(row["rung"], row["name"], f"avg_tokens={row['avg_tokens']:.2f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, rung in enumerate(bpe_ladder):
    merges, snapshots = train_bpe(rung["corpus"], rung["merges"])
    word = rung["eval"][0]
    pieces = apply_bpe(word, merges, continuation=True)
    matrix = np.zeros((len(pieces), len(word)))
    cursor = 0
    for i, piece in enumerate(pieces):
        clean = piece.replace("##", "")
        for j in range(cursor, min(len(word), cursor + len(clean))):
            matrix[i, j] = 1
        cursor += len(clean)
    show_matrix(axes[0, idx], matrix, f"D{idx + 1} {word}", list(word), pieces)

x = [row["rung"] for row in bpe_results]
y = [row["avg_tokens"] for row in bpe_results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_title("Average tokens per word")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("lower is shorter")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: train/serve vocabulary mismatch and missing continuation markers

In [ ]:

d5 = bpe_ladder[-1]
train_merges, snapshots = train_bpe(d5["corpus"], d5["merges"])
wrong_merges, wrong_snapshots = train_bpe(list(reversed(d5["corpus"])), d5["merges"])
word = "saopauloinvoice"
right = apply_bpe(word, train_merges, continuation=True)
wrong = apply_bpe(word, wrong_merges, continuation=False)
fixed = apply_bpe(word, train_merges, continuation=True)

print("right frozen vocab", right)
print("wrong retrained ids without markers", wrong)
print("fixed with markers", fixed)
print("marker count", sum(piece.startswith("##") for piece in fixed))


## Evaluate it + Practice

- Metric: average tokens per word, with lower values meaning shorter sequences
- No-skill baseline: word-level unknown token for every unseen form
- Cheap sanity check: D1 first merge must be (l,o), and low must shrink from 4 to 3 positions
- Ablation: retrain merges on serving data or drop ## markers and token identities change
- Failure signals: unstable ids, sudden length jumps, or detokenization that loses word boundaries

### Practice

**Practice:** Increase D3 merges by two and measure whether playing gets shorter.

**Practice:** Add a new multilingual-looking word and compare character length to BPE length.